# Phase 5 fix — LP-FT (Linear-Probe Fine-Tuning) on PlantDoc

**Diagnosis (step-wise Grad-CAM audit):**

| Stage | Acc | Attention |
|---|---|---|
| PlantVillage | 99.8% | leaf ✅ |
| Paddy Doctor | 97.0% | lesion ✅ |
| PlantDoc (full fine-tune) | 72.3% | background ❌ |

Full fine-tuning on the small, cluttered PlantDoc set **distorts** the healthy features from stages 1-2 (Kumar et al., ICLR 2022). **LP-FT** = freeze the Paddy backbone, train only a fresh 27-class head. Literature (Frontiers 2026) reports frozen probes beating full fine-tune by 11-15pp on field PlantDoc.

This notebook freezes the **OLD Paddy backbone** (the healthy 97% one — NOT the failed R model) and pushes the result to a NEW repo `iks-disease-plantdoc-lpft`, leaving the original `iks-disease-plantdoc` untouched for comparison.

**Cell 4 is the test cell** — accuracy (OLD vs LP-FT) + Grad-CAM, same format as the diagnosis. If LP-FT recovers accuracy and moves attention to the leaf → it's the fix. If not → we go to detect-then-crop.

In [ ]:
# Cell 2 — clean clone + deps + HF login + GPU check.
# GIT_LFS_SKIP_SMUDGE=1 dodges the LFS bandwidth stall on the .pt files.
import os, shutil, subprocess, sys

REPO_PATH = "/content/iks-rag-thesis"
REPO_URL = "https://github.com/ankit8453/iks-rag-thesis.git"

os.chdir("/content")  # step out before deleting REPO_PATH
shutil.rmtree(REPO_PATH, ignore_errors=True)
env = os.environ.copy()
env["GIT_LFS_SKIP_SMUDGE"] = "1"
r = subprocess.run(["git", "clone", REPO_URL, REPO_PATH], env=env, capture_output=True, text=True)
if r.returncode != 0:
    print(r.stdout); print(r.stderr); raise RuntimeError(f"git clone failed ({r.returncode})")
os.chdir(REPO_PATH); sys.path.insert(0, REPO_PATH)
print("Repo at:", os.getcwd())

DEPS = [
    "timm>=1.0", "albumentations>=1.4", "datasets>=2.20",
    "huggingface_hub>=0.24", "grad-cam>=1.5", "pydantic>=2.7",
    "opencv-python-headless", "matplotlib>=3.7",
]
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *DEPS], capture_output=True, text=True)
if r.returncode != 0:
    print("\n".join(r.stdout.splitlines()[-30:])); print("\n".join(r.stderr.splitlines()[-30:]))
    raise RuntimeError(f"pip install failed ({r.returncode})")
print("deps installed")

from huggingface_hub import HfApi, login
login()
print("HF user:", HfApi().whoami().get("name"))

import torch
print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Switch runtime to T4 GPU."

## Cell 3 — Train: freeze Paddy backbone, train PlantDoc head only

Resumable — pushes `checkpoint_latest.pt` to `iks-disease-plantdoc-lpft` every epoch, so a Colab timeout is harmless (re-run this cell to continue). Head-only training is light; ~25 epochs on T4.

In [ ]:
from src.disease.train import (
    CheckpointManager, _build_loaders_from_hf, auto_batch_size,
)
from src.disease.train_lpft import (
    DEFAULT_LPFT_REPO, build_lpft_model, train_lpft, PLANTDOC_NUM_CLASSES,
)
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
bs = auto_batch_size(380)
print(f"device={device} batch_size={bs}")

# PlantDoc train/val loaders (HF), 27 classes.
train_loader, val_loader, num_classes = _build_loaders_from_hf(
    "finetune_plantdoc", batch_size=bs, num_workers=2,
)
print(f"PlantDoc loaders ready — {num_classes} classes")

# Build LP-FT model: OLD Paddy backbone (frozen) + fresh 27-class head.
model = build_lpft_model(num_classes=PLANTDOC_NUM_CLASSES)

# Checkpoint manager → NEW repo (leaves the old plantdoc model untouched).
ckpt = CheckpointManager(DEFAULT_LPFT_REPO)
ckpt.ensure_repo(private=True)

# Resume if a partial LP-FT run exists.
prev = ckpt.try_load_latest()
start_epoch, history = 0, []
if prev is not None:
    start_epoch = int(prev.get("epoch", 0))
    history = list(prev.get("history", []))
    model.load_state_dict(prev["model_state"], strict=False)
    print(f"Resuming LP-FT from epoch {start_epoch}")

result = train_lpft(
    model, train_loader, val_loader, ckpt,
    num_classes=PLANTDOC_NUM_CLASSES,
    epochs=25, lr_head=1e-3, device=device,
    start_epoch=start_epoch, history=history,
)
print("\nBEST LP-FT val acc:", round(result["best_val_acc"], 4))

## Cell 4 — TEST: accuracy (OLD vs LP-FT) + Grad-CAM comparison

Same format as the diagnosis cells. Decision rule:
- LP-FT acc **up** (toward ~77%) AND heatmap **on the leaf** → ✅ the fix; build the paper on this.
- Otherwise → go to detect-then-crop (Step 2).

In [ ]:
import torch, random, numpy as np, matplotlib.pyplot as plt
from datasets import load_dataset
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from src.explain.gradcam import _preprocess_for_gradcam
from src.disease.infer import DiseaseInferenceEngine
from src.disease.train_lpft import DEFAULT_LPFT_REPO

pd_test = load_dataset("ankit-iiitdmj/iks-plantdoc", split="test")

def gt(row):
    return int(row["label_idx"]) if "label_idx" in row else int(row["label"])

random.seed(42)
idxs = random.sample(range(len(pd_test)), min(600, len(pd_test)))

MODELS = {
    "OLD (full fine-tune)": "ankit-iiitdmj/iks-disease-plantdoc",
    "LP-FT (frozen backbone)": DEFAULT_LPFT_REPO,
}
accs = {}
engines = {}
for name, repo in MODELS.items():
    eng = DiseaseInferenceEngine(model_source=repo, device="cuda")
    correct = sum(int(eng.predict(pd_test[i]["image"].convert("RGB")).prediction.class_index) == gt(pd_test[i]) for i in idxs)
    accs[name] = correct / len(idxs)
    engines[name] = eng
    print(f"{name:26s}: {accs[name]:.1%}  ({correct}/{len(idxs)})  [{eng.num_classes} cls]")

print(f"\nDelta (LP-FT - OLD): {(accs['LP-FT (frozen backbone)'] - accs['OLD (full fine-tune)'])*100:+.1f} pp")

# --- Grad-CAM: OLD vs LP-FT on the same cluttered images ---
def cam(eng, pil):
    tensor, rgb_u8, rgb_f = _preprocess_for_gradcam(pil, image_size=eng.image_size)
    mod = eng.model._module if hasattr(eng.model, "_module") else eng.model
    bb = eng.model.get_feature_extractor(); mod.eval()
    t = tensor.to(next(mod.parameters()).device).requires_grad_(True)
    pred = eng.predict(pil).prediction
    g = GradCAM(model=mod, target_layers=[bb.blocks[-2]])(
        input_tensor=t, targets=[ClassifierOutputTarget(int(pred.class_index))])[0]
    return show_cam_on_image(rgb_f, g, use_rgb=True), rgb_u8

random.seed(7)
for idx in random.sample(range(len(pd_test)), 4):
    pil = pd_test[idx]["image"].convert("RGB")
    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    ov_old, rgb = cam(engines["OLD (full fine-tune)"], pil)
    ax[0].imshow(rgb); ax[0].set_title("input (PlantDoc)"); ax[0].axis("off")
    ax[1].imshow(ov_old); ax[1].set_title(f"OLD full-FT ({accs['OLD (full fine-tune)']:.0%})"); ax[1].axis("off")
    ax[2].imshow(cam(engines["LP-FT (frozen backbone)"], pil)[0]); ax[2].set_title(f"LP-FT ({accs['LP-FT (frozen backbone)']:.0%})"); ax[2].axis("off")
    plt.tight_layout(); plt.show()